# TrendWaveletAELG Pure-Stack Study Analysis — v2

Analyzes the v2 refined search (24 configs per dataset vs 112 in v1).
Locked parameters: `latent_dim=16`, `trend_dim=3`, `active_g=false`.
Pruned wavelet list: `haar`, `db3`, `db4`, `coif2`, `coif3`, `sym3`.

**Key scientific questions:**
1. Which wavelet × basis_label combo wins across all 4 datasets?
2. Does v2 (latent_dim=16, trend_dim=3) outperform v1 (latent_dim=8, trend_dim=3/5)?
3. Does pure TrendWaveletAELG beat alternating TrendAELG + WaveletV3 stacks?
4. Is there a cross-dataset winner or does the best config vary by domain?

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 100)

ROOT = Path('experiments')
if not ROOT.exists():
    ROOT = Path('.')

RESULTS = ROOT / 'results'
REPORT_PATH = ROOT / 'analysis_reports' / 'trendwaveletaelg_pure_v2_study_analysis.md'

DATASET_ORDER = ['m4', 'tourism', 'traffic', 'weather']
DATASET_LABELS = {
    'm4': 'M4-Yearly',
    'tourism': 'Tourism-Yearly',
    'traffic': 'Traffic-96',
    'weather': 'Weather-96',
}

# v2 search CSVs (primary data)
V2_SEARCH_CSVS = {
    ds: RESULTS / ds / 'trendwaveletaelg_pure_v2_study_results.csv'
    for ds in DATASET_ORDER
}
V2_CROSS_CSV = RESULTS / 'trendwaveletaelg_pure_v2_cross_dataset_results.csv'

# v1 comparison CSVs
V1_SEARCH_CSVS = {
    ds: RESULTS / ds / 'trendwaveletaelg_pure_study_results.csv'
    for ds in DATASET_ORDER
}

# Alternating study (TrendAELG + WaveletV3AE) for pure-vs-alternating comparison
ALTERNATING_CSVS = {
    ds: RESULTS / ds / 'wavelet_v3ae_study_results.csv'
    for ds in DATASET_ORDER
}

In [ ]:
def load_csv(path: Path):
    if not path.exists():
        return None
    df = pd.read_csv(path)
    for c in ['search_round', 'best_val_loss', 'trend_dim_cfg', 'trend_thetas_dim_cfg',
              'latent_dim_cfg', 'basis_dim']:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    return df


# Load v2 data
v2_dfs = {ds: load_csv(p) for ds, p in V2_SEARCH_CSVS.items()}
v2_cross = load_csv(V2_CROSS_CSV)

# Load v1 data for comparison
v1_dfs = {ds: load_csv(p) for ds, p in V1_SEARCH_CSVS.items()}

# Load alternating data
alt_dfs = {ds: load_csv(p) for ds, p in ALTERNATING_CSVS.items()}

print('--- v2 (TrendWaveletAELG, latent_dim=16, trend_dim=3) ---')
for ds in DATASET_ORDER:
    df = v2_dfs[ds]
    if df is None:
        print(f'  {ds}: missing')
    else:
        rounds = sorted(df['search_round'].dropna().astype(int).unique().tolist())
        print(f'  {ds}: rows={len(df)} configs={df["config_name"].nunique()} rounds={rounds}')
print(f'  cross rows: {0 if v2_cross is None else len(v2_cross)}')

print('\n--- v1 (TrendWaveletAELG, latent_dim=8, trend_dim=[3,5]) ---')
for ds in DATASET_ORDER:
    df = v1_dfs[ds]
    if df is None:
        print(f'  {ds}: missing')
    else:
        rounds = sorted(df['search_round'].dropna().astype(int).unique().tolist())
        print(f'  {ds}: rows={len(df)} configs={df["config_name"].nunique()} rounds={rounds}')

print('\n--- Alternating (WaveletV3AE) ---')
for ds in DATASET_ORDER:
    df = alt_dfs[ds]
    if df is None:
        print(f'  {ds}: missing')
    else:
        rounds = sorted(df['search_round'].dropna().astype(int).unique().tolist())
        print(f'  {ds}: rows={len(df)} configs={df["config_name"].nunique()} rounds={rounds}')

## Section 1: Per-Dataset Round-by-Round Survival

Successive-halving funnel and top-10 leaderboards per dataset.

In [ ]:
report = []
report.append('# TrendWaveletAELG Pure-Stack Study Analysis — v2')
report.append('')
report.append('Fixed: `latent_dim=16`, `trend_dim=3`, `active_g=false`')
report.append('Search: 6 wavelet families × 4 basis labels = 24 configs per dataset')
report.append('')

report.append('# Section 1: Per-Dataset Round-by-Round Survival')
report.append('')

for ds in DATASET_ORDER:
    df = v2_dfs[ds]
    report.append(f'## {DATASET_LABELS[ds]}')
    report.append('')
    if df is None or df.empty:
        report.append(f'- Missing CSV: `{V2_SEARCH_CSVS[ds]}`')
        report.append('')
        continue

    report.append(f'- Rows: {len(df)}')
    report.append(f'- Unique configs: {df["config_name"].nunique()}')
    report.append('')

    # Successive-halving funnel
    report.append('### Successive-Halving Funnel')
    rounds = sorted(df['search_round'].dropna().astype(int).unique().tolist())
    funnel_rows = []
    prev = None
    for r in rounds:
        rdf = df[df['search_round'].astype('Int64') == r]
        n_cfg = rdf['config_name'].nunique()
        keep = '-' if prev is None else f'{n_cfg}/{prev} ({n_cfg/max(prev,1):.0%})'
        best = pd.to_numeric(rdf['best_val_loss'], errors='coerce').median()
        funnel_rows.append({'Round': r, 'Configs': n_cfg, 'Rows': len(rdf),
                            'Median best_val_loss': best, 'Kept': keep})
        prev = n_cfg
    report.append(pd.DataFrame(funnel_rows).to_markdown(index=False, floatfmt='.4f'))
    report.append('')

    # Round 3 leaderboard
    report.append('### Round 3 Leaderboard (Top 10)')
    r3 = df[df['search_round'].astype('Int64') == 3].copy()
    if r3.empty:
        report.append('Round 3 not yet available.')
        report.append('')
    else:
        td_col = 'trend_dim_cfg' if 'trend_dim_cfg' in r3.columns else 'trend_thetas_dim_cfg'
        grp = (
            r3.groupby(['config_name', 'wavelet_type', 'bd_label', td_col, 'latent_dim_cfg'], dropna=False)
            .agg(median_best_val_loss=('best_val_loss', 'median'),
                 std_best_val_loss=('best_val_loss', 'std'),
                 n=('best_val_loss', 'count'))
            .sort_values('median_best_val_loss')
            .head(10)
            .reset_index()
        )
        report.append(grp.to_markdown(index=False, floatfmt='.4f'))
        report.append('')

print('Section 1 done')
print('\n'.join(report[:40]))

## Section 2: Wavelet Family Marginals

Heatmap (basis_label × wavelet_type) and marginal medians per dataset.

In [ ]:
report.append('# Section 2: Wavelet Family Marginals')
report.append('')

WAVELET_ORDER = ['haar', 'db3', 'db4', 'coif2', 'coif3', 'sym3']
BASIS_ORDER = ['eq_fcast', 'lt_fcast', 'eq_bcast', 'lt_bcast']

for ds in DATASET_ORDER:
    df = v2_dfs[ds]
    report.append(f'## {DATASET_LABELS[ds]}')
    report.append('')
    if df is None or df.empty:
        report.append('No data.')
        report.append('')
        continue

    # Use round 1 for full-space marginals (all 24 configs present)
    r1 = df[df['search_round'].astype('Int64') == 1].copy()
    if r1.empty:
        report.append('Round 1 not available.')
        report.append('')
        continue

    # Heatmap: basis_label x wavelet_type median val_loss
    report.append('### Heatmap: Median val_loss by basis_label × wavelet_type (Round 1)')
    bd_col = 'bd_label' if 'bd_label' in r1.columns else 'basis_label'
    pivot = (
        r1.groupby([bd_col, 'wavelet_type'])['best_val_loss']
        .median()
        .unstack('wavelet_type')
        .reindex(index=BASIS_ORDER, columns=WAVELET_ORDER)
    )
    report.append(pivot.to_markdown(floatfmt='.4f'))
    report.append('')

    # Marginal by wavelet_type
    report.append('### Marginal by wavelet_type (Round 1)')
    wt_mg = (
        r1.groupby('wavelet_type')['best_val_loss']
        .agg(['median', 'mean', 'count'])
        .rename(columns={'median': 'median_val_loss', 'mean': 'mean_val_loss', 'count': 'n'})
        .reindex(WAVELET_ORDER)
        .sort_values('median_val_loss')
    )
    report.append(wt_mg.to_markdown(floatfmt='.4f'))
    report.append('')

    # Marginal by basis_label
    report.append('### Marginal by basis_label (Round 1)')
    bl_mg = (
        r1.groupby(bd_col)['best_val_loss']
        .agg(['median', 'mean', 'count'])
        .rename(columns={'median': 'median_val_loss', 'mean': 'mean_val_loss', 'count': 'n'})
        .reindex(BASIS_ORDER)
        .sort_values('median_val_loss')
    )
    report.append(bl_mg.to_markdown(floatfmt='.4f'))
    report.append('')

print('Section 2 done')

## Section 3: v2 vs v1 Comparison

Does `latent_dim=16` + `trend_dim=3` (v2) beat v1 (`latent_dim=8`, `trend_dim=[3,5]`)?
Compare on the 6 wavelet families that appear in both.

In [ ]:
report.append('# Section 3: v2 vs v1 Comparison')
report.append('')
report.append('Comparing `latent_dim=16, trend_dim=3` (v2) vs `latent_dim=8, trend_dim=3` (v1 best subset).')
report.append('')

SHARED_WAVELETS = ['haar', 'db3', 'db4', 'coif2', 'coif3', 'sym3']

for ds in DATASET_ORDER:
    report.append(f'## {DATASET_LABELS[ds]}')
    report.append('')

    v2_df = v2_dfs[ds]
    v1_df = v1_dfs[ds]

    if v2_df is None:
        report.append('v2 data missing.')
        report.append('')
        continue

    if v1_df is None:
        report.append('v1 data missing (comparison unavailable).')
        report.append('')
        continue

    # Use round 3 if available, else best available
    def best_round(df):
        max_r = df['search_round'].dropna().max()
        return df[df['search_round'].astype('Int64') == int(max_r)].copy()

    v2_r = best_round(v2_df)
    v1_r = best_round(v1_df)

    # Filter v1 to trend_dim=3 and shared wavelet families
    td_col = 'trend_dim_cfg' if 'trend_dim_cfg' in v1_r.columns else 'trend_thetas_dim_cfg'
    v1_r_td3 = v1_r[
        (v1_r[td_col] == 3) & (v1_r['wavelet_type'].isin(SHARED_WAVELETS))
    ].copy()

    v2_med = v2_r.groupby('wavelet_type')['best_val_loss'].median().rename('v2_ld16_td3')
    v1_med = v1_r_td3.groupby('wavelet_type')['best_val_loss'].median().rename('v1_ld8_td3')

    cmp = pd.concat([v2_med, v1_med], axis=1).dropna()
    cmp['diff (v2-v1)'] = cmp['v2_ld16_td3'] - cmp['v1_ld8_td3']
    cmp['winner'] = cmp.apply(
        lambda r: 'v2' if r['diff (v2-v1)'] < 0 else ('v1' if r['diff (v2-v1)'] > 0 else 'tie'), axis=1
    )
    report.append(cmp.sort_values('diff (v2-v1)').to_markdown(floatfmt='.4f'))
    report.append('')

    if not cmp.empty:
        wins = cmp['winner'].value_counts()
        report.append(f'Win/Loss: v2={wins.get("v2", 0)}, v1={wins.get("v1", 0)}, tie={wins.get("tie", 0)}')
        if len(cmp) >= 5:
            stat, pval = stats.wilcoxon(cmp['v2_ld16_td3'], cmp['v1_ld8_td3'])
            report.append(f'Wilcoxon signed-rank: stat={stat:.4f}, p={pval:.4f}')
    report.append('')

print('Section 3 done')

## Section 4: Pure vs Alternating Comparison

Does the combined TrendWaveletAELG block (pure stack) match or beat
the alternating TrendAELG + WaveletV3AE approach?

In [ ]:
report.append('# Section 4: Pure vs Alternating Comparison')
report.append('')
report.append('Pure = TrendWaveletAELG pure stack (v2). Alternating = TrendAELG + WaveletV3AE.')
report.append('')

WAVELET_TYPE_TO_BLOCK = {
    'haar': 'HaarWaveletV3AE', 'db3': 'DB3WaveletV3AE', 'db4': 'DB4WaveletV3AE',
    'coif2': 'Coif2WaveletV3AE', 'coif3': 'Coif3WaveletV3AE', 'sym3': 'Symlet3WaveletV3AE',
}
BLOCK_TO_WAVELET = {v: k for k, v in WAVELET_TYPE_TO_BLOCK.items()}

for ds in DATASET_ORDER:
    report.append(f'## {DATASET_LABELS[ds]}')
    report.append('')

    pure_df = v2_dfs[ds]
    alt_df = alt_dfs[ds]

    if pure_df is None or alt_df is None:
        report.append('Not enough data for comparison.')
        report.append('')
        continue

    # Best available round
    pure_max_r = pure_df['search_round'].dropna().max()
    alt_max_r = alt_df['search_round'].dropna().max()

    pure_r = pure_df[pure_df['search_round'].astype('Int64') == int(pure_max_r)].copy()
    alt_r = alt_df[alt_df['search_round'].astype('Int64') == int(alt_max_r)].copy()

    pure_overall = pure_r['best_val_loss'].median()
    alt_overall = alt_r['best_val_loss'].median()
    report.append(f'- Pure TrendWaveletAELG median val_loss: {pure_overall:.4f} (round {int(pure_max_r)})')
    report.append(f'- Alternating WaveletV3AE median val_loss: {alt_overall:.4f} (round {int(alt_max_r)})')
    report.append(f'- Difference (pure - alt): {pure_overall - alt_overall:.4f}')
    report.append('')

    # Per-wavelet-family comparison (where alternating data has wavelet_family column)
    if 'wavelet_family' in alt_r.columns:
        report.append('### Per-Wavelet-Family Comparison')
        pure_wf = pure_r.groupby('wavelet_type')['best_val_loss'].median().rename('pure')
        alt_r['wt_mapped'] = alt_r['wavelet_family'].map(BLOCK_TO_WAVELET)
        alt_wf = alt_r.groupby('wt_mapped')['best_val_loss'].median().rename('alternating')
        cmp = pd.concat([pure_wf, alt_wf], axis=1).dropna()
        cmp['diff (pure-alt)'] = cmp['pure'] - cmp['alternating']
        cmp = cmp.sort_values('diff (pure-alt)')
        report.append(cmp.to_markdown(floatfmt='.4f'))
        report.append('')

        pure_wins = (cmp['diff (pure-alt)'] < 0).sum()
        alt_wins = (cmp['diff (pure-alt)'] > 0).sum()
        report.append(f'Pure wins: {pure_wins}, Alternating wins: {alt_wins}')
        report.append('')

print('Section 4 done')

## Section 5: Cross-Dataset Robustness Leaderboard

Which configs survive well across all 4 datasets?

In [ ]:
report.append('# Section 5: Cross-Dataset Robustness Leaderboard')
report.append('')

# Use cross-dataset CSV if available
if v2_cross is not None and not v2_cross.empty:
    metric = 'best_val_loss' if 'best_val_loss' in v2_cross.columns else 'final_val_loss'
    board = (
        v2_cross.groupby(['canonical_config_id', 'source_datasets'], dropna=False)
        .agg(mean_metric=(metric, 'mean'), std_metric=(metric, 'std'), n=(metric, 'count'))
        .sort_values('mean_metric')
        .reset_index()
    )
    report.append('## Cross-Dataset Benchmark Results')
    report.append(board.to_markdown(index=False, floatfmt='.4f'))
    report.append('')
else:
    report.append('Cross-dataset CSV not yet generated. Building from per-dataset round-3 results.')
    report.append('')

    # Build a cross-dataset view from individual round-3 results
    per_ds_top = {}
    for ds in DATASET_ORDER:
        df = v2_dfs[ds]
        if df is None:
            continue
        max_r = df['search_round'].dropna().max()
        r = df[df['search_round'].astype('Int64') == int(max_r)].copy()
        grp = (
            r.groupby('wavelet_type')['best_val_loss']
            .median()
            .rename(DATASET_LABELS[ds])
        )
        per_ds_top[ds] = grp

    if per_ds_top:
        cross = pd.concat(per_ds_top.values(), axis=1)
        cross['mean'] = cross.mean(axis=1)
        cross['std'] = cross.std(axis=1)
        cross = cross.sort_values('mean')
        report.append('## Per-Wavelet-Type Cross-Dataset Summary (Round 3 Medians)')
        report.append(cross.to_markdown(floatfmt='.4f'))
        report.append('')

        # Also basis_label cross-dataset
        per_ds_bl = {}
        for ds in DATASET_ORDER:
            df = v2_dfs[ds]
            if df is None:
                continue
            max_r = df['search_round'].dropna().max()
            r = df[df['search_round'].astype('Int64') == int(max_r)].copy()
            bd_col = 'bd_label' if 'bd_label' in r.columns else 'basis_label'
            grp = r.groupby(bd_col)['best_val_loss'].median().rename(DATASET_LABELS[ds])
            per_ds_bl[ds] = grp

        cross_bl = pd.concat(per_ds_bl.values(), axis=1)
        cross_bl['mean'] = cross_bl.mean(axis=1)
        cross_bl = cross_bl.sort_values('mean')
        report.append('## Per-Basis-Label Cross-Dataset Summary')
        report.append(cross_bl.to_markdown(floatfmt='.4f'))
        report.append('')

print('Section 5 done')

## Section 6: Hyperparameter Sensitivity

Full marginals across all round-1 data (all 24 configs visible).

In [ ]:
report.append('# Section 6: Hyperparameter Sensitivity (Round 1 Marginals)')
report.append('')

for ds in DATASET_ORDER:
    df = v2_dfs[ds]
    report.append(f'## {DATASET_LABELS[ds]}')
    report.append('')
    if df is None or df.empty:
        report.append('No data.')
        report.append('')
        continue

    r1 = df[df['search_round'].astype('Int64') == 1].copy()
    if r1.empty:
        report.append('No round-1 data.')
        report.append('')
        continue

    bd_col = 'bd_label' if 'bd_label' in r1.columns else 'basis_label'
    for col in ['wavelet_type', bd_col]:
        if col not in r1.columns:
            continue
        mg = (
            r1.groupby(col)
            .agg(median_val_loss=('best_val_loss', 'median'),
                 mean_val_loss=('best_val_loss', 'mean'),
                 std_val_loss=('best_val_loss', 'std'),
                 n=('best_val_loss', 'count'))
            .sort_values('median_val_loss')
            .reset_index()
        )
        report.append(f'### Marginal by {col}')
        report.append(mg.to_markdown(index=False, floatfmt='.4f'))
        report.append('')

print('Section 6 done')

In [ ]:
# Write final markdown report
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text('\n'.join(report), encoding='utf-8')
print(f'Wrote report to: {REPORT_PATH}')
print(f'Total lines: {len(report)}')
print('\n'.join(report[:30]))